# Video Sign Confusion Circos Diagram

Aggregate the per-subject confusion matrices produced by `videos/scripts/evaluate_cv.py` (`confusion_matrix_subject_*.pkl`) into a single 50×50 matrix, compute mean ± std F1 / accuracy across folds, and render a pycirclize chord diagram of cross-class confusions. The class mapping is the canonical 50-sign Spanish dictionary used throughout the video pipeline (see `videos/src/evaluate.py:CLASS_MAPPING`).

## Configuration

In [ ]:
PER_SUBJECT_PKL_GLOB = "*subject_*.pkl"
N_SUBJECTS = 5


## Class mapping

In [ ]:
import pickle
import pandas as pd
import json
from pycirclize import Circos
import glob
import seaborn as sns
import numpy as np

cor = {0: 'Niña', 1: 'Poco', 2: 'Feliz', 3: 'Hola', 4: 'Sentir', 5: 'Contento', 6: 'Jucioso', 7: 'Diferente', 8: 'Abuelo', 9: 'Hombre', 10: 'Seña', 
        11: 'Permiso', 12: 'Mal', 13: 'Con Gusto', 14: 'Ustedes', 15: 'Perdon', 16: '¿Como estas?', 17: 'Tio ', 18: 'Hermano', 19: 'Nunca', 20: 'Trabajar', 
        21: 'Aburrido', 22: 'Vivir', 23: 'Buenos Dias', 24: 'Todos', 25: 'Bienvenido', 26: '¿Como?', 27: '¿Quien?', 28: 'Por favor', 29: '¿Cuando?', 30: 'Tu', 
        31: 'Buenas Tardes', 32: 'Gracias', 33: 'Hambre', 34: 'Niño', 35: 'Mujer', 36: 'Adios', 37: 'Personas', 38: 'Bien', 39: 'Yo', 40: 'Triste', 
        41: '¿Por que?', 42: 'Mucho', 43: 'Nombre', 44: '¿Que?', 45: 'Comer', 46: 'Buenas Noches', 47: '¿Donde?', 48: 'Familia', 49: 'Mas o menos'}


## Aggregate per-subject confusion matrices

In [ ]:
agg_cm = np.zeros((50,50))
from sklearn.metrics import f1_score, accuracy_score

f1_scores = []
accuracy_scores = []

for file in glob.glob(PER_SUBJECT_PKL_GLOB):
    with open(file, 'rb') as f:
        handle = pickle.load(f)
        y_true = handle['real_labels']
        y_pred = handle['inferences']
        cm = handle['confusion_matrix']
        f1_scores.append(f1_score(y_true, y_pred, average='weighted'))
        accuracy_scores.append(accuracy_score(y_true, y_pred))

    agg_cm += cm

np.fill_diagonal(agg_cm, 0)
agg_cm = agg_cm / N_SUBJECTS

matrix_df = pd.DataFrame(agg_cm, index=[cor[i].upper() for i in range(50)], columns=[cor[i].upper() for i in range(50)])
circos = Circos.chord_diagram(
    matrix_df,
    space=3,
    label_kws=dict(size=12,orientation="vertical"),
    link_kws=dict(ec = "black", lw=0.5, direction=1),
    order="desc",
)

circos.savefig("confusion_matrix_transformer_circos.png", dpi=300)


## Per-fold metrics

In [ ]:
np.mean(f1_scores),np.std(f1_scores)

In [ ]:
np.mean(accuracy_scores),np.std(accuracy_scores)

## Reconstructed macro F1 from aggregated matrix

In [ ]:
from sklearn.metrics import f1_score

# Get true labels and predicted labels from the confusion matrix
y_true = []
y_pred = []
for i in range(agg_cm.shape[0]):
	for j in range(agg_cm.shape[1]):
		count = int(round(agg_cm[i, j]))
		y_true.extend([i] * count)
		y_pred.extend([j] * count)
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate F1 score (macro average)
f1 = f1_score(y_true, y_pred, average='macro')
print("Macro F1 score:", f1)